# Handy Mouse By Lakshit Saini

In [1]:
import cv2
import numpy as np
import mediapipe as mp
import pyautogui
import time

In [2]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.8, max_num_hands=2)

In [3]:
screen_w, screen_h = pyautogui.size()

In [4]:
prev_x, prev_y = None, None
clicking = False
dragging = False
screenshot_taken = False
screenshot_cooldown = 2
last_screenshot_time = time.time()
scroll_active = False

In [5]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)  
    h, w, _ = frame.shape
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)

    hand_landmarks_list = []

    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            hand_landmarks_list.append(hand_landmarks)

    
    if hand_landmarks_list:
        hand_landmarks = hand_landmarks_list[0]  
        index_tip = hand_landmarks.landmark[8]  

        
        cursor_x = int(index_tip.x * screen_w)
        cursor_y = int(index_tip.y * screen_h)

        
        if prev_x is not None and prev_y is not None:
            cursor_x = int(0.8 * prev_x + 0.2 * cursor_x)
            cursor_y = int(0.8 * prev_y + 0.2 * cursor_y)

        pyautogui.moveTo(cursor_x, cursor_y)

        prev_x, prev_y = cursor_x, cursor_y

    
    if hand_landmarks_list:
        hand_landmarks = hand_landmarks_list[0]
        index_tip = hand_landmarks.landmark[8]
        index_base = hand_landmarks.landmark[5]
        middle_tip = hand_landmarks.landmark[12]
        thumb_tip = hand_landmarks.landmark[4]

        # Left Click (Agar index finger hawa mein uth jaaye)
        if index_tip.y < index_base.y and middle_tip.y > index_base.y:
            if not clicking:
                pyautogui.click()
                clicking = True
        else:
            clicking = False  

        # Right Click (Agar middle finger hawa mein uth jaaye)
        if middle_tip.y < index_base.y and index_tip.y > index_base.y:
            pyautogui.rightClick()

        # Double Click (Dono index + middle finger uth jaaye)
        if index_tip.y < index_base.y and middle_tip.y < index_base.y:
            pyautogui.doubleClick()

        # Drag and Drop (Agar index finger aur thumb mil gaye)
        pinch_distance = np.linalg.norm(np.array([index_tip.x, index_tip.y]) - np.array([thumb_tip.x, thumb_tip.y]))

        if pinch_distance < 0.03:
            if not dragging:
                pyautogui.mouseDown()
                dragging = True
        else:
            if dragging:
                pyautogui.mouseUp()
                dragging = False

        # Scrolling (Hath upar neeche karo, scroll ho jayega )
        palm_y = hand_landmarks.landmark[0].y  
        if palm_y < 0.3:
            pyautogui.scroll(5)  
            scroll_active = True
        elif palm_y > 0.7:
            pyautogui.scroll(-5)  
            scroll_active = True
        else:
            scroll_active = False  

    # Zoom In/Out (Dono hath door le jao ya paas lao)
    if len(hand_landmarks_list) == 2:
        hand1, hand2 = hand_landmarks_list[0], hand_landmarks_list[1]
        index_tip_1 = hand1.landmark[8]
        index_tip_2 = hand2.landmark[8]

        x1, y1 = int(index_tip_1.x * w), int(index_tip_1.y * h)
        x2, y2 = int(index_tip_2.x * w), int(index_tip_2.y * h)

        distance = np.linalg.norm(np.array([x1, y1]) - np.array([x2, y2]))

        if distance > 100:  
            pyautogui.hotkey("ctrl", "+")  # Zoom in karo
        elif distance < 50:  
            pyautogui.hotkey("ctrl", "-")  # Zoom out karo

        #Screenshot Gesture (Dono index finger cross kar do)
        if distance < 30:  
            current_time = time.time()
            if not screenshot_taken and current_time - last_screenshot_time > screenshot_cooldown:
                pyautogui.screenshot("screenshot.png")
                print("📸 Screenshot Le Liya Bhai!")
                last_screenshot_time = current_time
                screenshot_taken = True
        else:
            screenshot_taken = False  

    
    cv2.putText(frame, "✋ Hath Hila: Cursor | ☝ Index Upar: Click | ✌ Middle Upar: Right Click | 🤞 2 Fingers: Double Click",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    cv2.putText(frame, "👌 Pinch Index+Thumb: Drag & Drop | 🤲 Hath Upar/Neeche: Scroll | 🔍 Zoom: Dono Hath Door/Paas",
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    cv2.putText(frame, "✖ Cross Index Fingers: Screenshot 📸", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    
    cv2.imshow("Gesture-Controlled Mouse", frame)

   
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()


📸 Screenshot Le Liya Bhai!


In [6]:
cap.release()
cv2.destroyAllWindows()